# Managed Spot Training for XGBoost (V3)


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

---

> **SageMaker Python SDK v3 note:** This notebook has been migrated to the SageMaker Python SDK **v3**. Managed Spot Training for XGBoost now uses `ModelTrainer` (from `sagemaker-train`) instead of the v2 `Estimator`/`sagemaker.xgboost.XGBoost`. Spot training is configured through the structured `Compute` (`enable_managed_spot_training=True`), `StoppingCondition` (`max_runtime_in_seconds` / `max_wait_time_in_seconds`), and `CheckpointConfig` (`s3_uri`) config objects. Automatic Model Tuning uses the v3 `HyperparameterTuner` (from `sagemaker.train.tuner`) with parameter ranges from `sagemaker.core.parameter`, and container/session helpers come from `sagemaker.core`. Install with `pip install sagemaker` (v3).

---


This notebook shows usage of SageMaker Managed Spot infrastructure for XGBoost training. Below we show how Spot instances can be used for the 'algorithm mode' and 'script mode' training methods with the XGBoost container. 

[Managed Spot Training](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html) uses Amazon EC2 Spot instance to run training jobs instead of on-demand instances. You can specify which training jobs use spot instances and a stopping condition that specifies how long Amazon SageMaker waits for a job to run using Amazon EC2 Spot instances.

This notebook was tested in Amazon SageMaker Studio on a ml.t3.medium instance with Python 3 (Data Science) kernel.

In this notebook we will perform XGBoost training as described [here](). See the original notebook for more details on the data. 

### Setup variables and define functions

In [1]:
# [exec copy] pip install disabled (env pre-provisioned)
pass

In [2]:
%%time

import io
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

# [exec copy] explicit region/role/session for out-of-Studio papermill run
boto_session = boto3.Session(region_name="us-west-1")
sagemaker_session = Session(boto_session=boto_session)
role = get_execution_role(sagemaker_session=sagemaker_session)
region = sagemaker_session.boto_region_name

bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/DEMO-xgboost-spot"
default_bucket_prefix = sagemaker_session.default_bucket_prefix

if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"


[07/13/26 16:10:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=3551467;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=3551468;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


CPU times: user 1.19 s, sys: 363 ms, total: 1.56 s
Wall time: 3.5 s


### Fetching the dataset

In [3]:
%%time
s3 = boto3.client("s3")
# Load the dataset
FILE_DATA = "abalone"
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    f"datasets/tabular/uci_abalone/abalone.libsvm",
    FILE_DATA,
)
sagemaker_session.upload_data(FILE_DATA, bucket=bucket, key_prefix=prefix + "/train")
sagemaker_session.upload_data(FILE_DATA, bucket=bucket, key_prefix=prefix + "/validation")

[07/13/26 16:10:42] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=3551473;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=3551474;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

CPU times: user 105 ms, sys: 70.1 ms, total: 175 ms
Wall time: 612 ms


's3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-spot/validation/abalone'

### Obtaining the latest XGBoost container
We obtain the new container by specifying the framework version (1.7-1). This version specifies the upstream XGBoost framework version (1.7) and an additional SageMaker version (1). If you have an existing XGBoost workflow based on the previous (1.0-1, 1.2-2, 1.3-1 or 1.5-1) container, this would be the only change necessary to get the same workflow working with the new container.

In v3, the `image_uris` helper lives under `sagemaker.core`.

In [4]:
from sagemaker.core import image_uris

container = image_uris.retrieve("xgboost", region, "1.7-1")

[07/13/26 16:10:43] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=3551479;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=3551480;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=3551487;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3551488;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#535\535]8;;\

### Training the XGBoost model

After setting training parameters, we kick off training, and poll for status until training is completed, which in this example, takes few minutes.

To run training on SageMaker in v3 we construct a `ModelTrainer` (from `sagemaker-train`), which replaces the v2 `Estimator`. It accepts structured configuration objects:

* __training_image__: The XGBoost container image URI retrieved above.
* __role__: Role ARN.
* __hyperparameters__: A dictionary passed to the algorithm as hyperparameters.
* __compute__: A `Compute` object describing the instance type/count (and, below, the managed spot training flag). __Note__: This particular mode does not currently support training on GPU instance types.
* __sagemaker_session__ *(optional)*: The session used to train on SageMaker.

If Spot instances are used, the training job can be interrupted, causing it to take longer to start or finish. If a training job is interrupted, a checkpointed snapshot can be used to resume from a previously saved point and can save training time (and cost).

To enable checkpointing for Managed Spot Training using SageMaker XGBoost we configure three things through v3 config objects: 

1. Set `Compute(enable_managed_spot_training=True)` - the structured v3 equivalent of the v2 `use_spot_instances` boolean. 

2. Set `StoppingCondition(max_wait_time_in_seconds=...)` - the amount of time you are willing to wait for Spot infrastructure to become available. Some instance types are harder to get at Spot prices and you may have to wait longer. You are not charged for time spent waiting for Spot infrastructure to become available, you're only charged for actual compute time spent once Spot instances have been successfully procured. 

3. Set `CheckpointConfig(s3_uri=...)` - this tells SageMaker an S3 location where to save checkpoints. While not strictly necessary, checkpointing is highly recommended for Managed Spot Training jobs due to the fact that Spot instances can be interrupted with short notice and using checkpoints to resume from the last interruption ensures you don't lose any progress made before the interruption.

Feel free to toggle the `use_spot_instances` variable to see the effect of running the same job using regular (a.k.a. "On Demand") infrastructure.

Note that `max_wait_time_in_seconds` can be set if and only if managed spot training is enabled and must be greater than or equal to `max_runtime_in_seconds`.

In [5]:
import time
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    StoppingCondition,
    CheckpointConfig,
    OutputDataConfig,
    InputData,
)

hyperparameters = {
    "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    "min_child_weight": "6",
    "subsample": "0.7",
    "objective": "reg:squarederror",
    "num_round": "50",
    "verbosity": "2",
}

instance_type = "ml.m5.4xlarge"
output_path = "s3://{}/{}/{}/output".format(bucket, prefix, "abalone-xgb")
content_type = "libsvm"

job_name = "DEMO-xgboost-spot-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
print("Training job", job_name)

use_spot_instances = True
max_run = 3600
max_wait = 7200 if use_spot_instances else None
checkpoint_s3_uri = (
    "s3://{}/{}/checkpoints/{}".format(bucket, prefix, job_name) if use_spot_instances else None
)
print("Checkpoint path:", checkpoint_s3_uri)

# Managed spot training + stopping condition are expressed through structured v3 configs.
compute = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=5,  # 5 GB
    enable_managed_spot_training=use_spot_instances,
)
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=max_run,
    max_wait_time_in_seconds=max_wait,
)
checkpoint_config = (
    CheckpointConfig(s3_uri=checkpoint_s3_uri) if use_spot_instances else None
)

model_trainer = ModelTrainer(
    training_image=container,
    role=role,
    sagemaker_session=sagemaker_session,
    hyperparameters=hyperparameters,
    compute=compute,
    stopping_condition=stopping_condition,
    checkpoint_config=checkpoint_config,
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    base_job_name=job_name,
)

train_input = InputData(
    channel_name="train",
    data_source="s3://{}/{}/{}".format(bucket, prefix, "train"),
    content_type="libsvm",
)
model_trainer.train(input_data_config=[train_input])

Training job DEMO-xgboost-spot-2026-07-13-23-10-44
Checkpoint path: s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-spot/checkpoints/DEMO-xgboost-spot-2026-07-13-23-10-44


[07/13/26 16:10:44] INFO     Role 'arn:aws:iam::729646638167:role/Admin' validated for     ]8;id=3551495;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3551496;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             training. Using it.                                                                   

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=3551503;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3551504;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#204\204]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=3551511;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3551512;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             746614075791.dkr.ecr.us-west-1.amazonaws.com/sagemaker-xgboost:1.                     
                             7-1                                                                                   

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3551519;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3551520;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

...

                    INFO     DEMO-xgboost-spot-2026-07-13-23-10-44-20260713161044/algo-1-1783984 ]8;id=3552227;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3552228;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             292:                                                                                  
                             [23:12:55] INFO: ../src/tree/updater_prune.cc:98: tree pruning end,                   
                             30 extra nodes, 0 pruned nodes, max_depth=5                                           

                    INFO     DEMO-xgboost-spot-2026-07-13-23-10-44-20260713161044/algo-1-1783984 ]8;id=3552233;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3552234;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             292:                                                                                  
                             [48]#011train-rmse:1.71716                                                            

                    INFO     DEMO-xgboost-spot-2026-07-13-23-10-44-20260713161044/algo-1-1783984 ]8;id=3552239;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3552240;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             292:                                                                                  
                             [23:12:55] INFO: ../src/tree/updater_prune.cc:98: tree pruning end,                   
                             42 extra nodes, 0 pruned nodes, max_depth=5                                           

                    INFO     DEMO-xgboost-spot-2026-07-13-23-10-44-20260713161044/algo-1-1783984 ]8;id=3552245;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3552246;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             292:                                                                                  
                             [49]#011train-rmse:1.70623                                                            

[07/13/26 16:13:12] INFO     Final Resource Status: Completed                                    ]8;id=3552252;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3552253;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31475\31475]8;;\

### Savings
Towards the end of the job you should see two lines of output printed:

- `Training seconds: X` : This is the actual compute-time your training job spent
- `Billable seconds: Y` : This is the time you will be billed for after Spot discounting is applied.

If you enabled managed spot training, then you should see a notable difference between `X` and `Y` signifying the cost savings you will get for having chosen Managed Spot Training. This should be reflected in an additional line:
- `Managed Spot Training savings: (1-Y/X)*100 %`

### Train with Automatic Model Tuning ([HPO](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)) <a id='AMT'></a> and Spot Training enabled 
***
You could also train with Amazon SageMaker Automatic Model Tuning. AMT, also known as hyperparameter tuning, finds the best version of a model by running many training jobs on your dataset using the algorithm and ranges of hyperparameters that you specify. It then chooses the hyperparameter values that result in a model that performs the best, as measured by a metric that you choose. We will use the v3 `HyperparameterTuner` (from `sagemaker.train.tuner`) object to interact with Amazon SageMaker hyperparameter tuning APIs.
        
The code sample below shows you how to use the v3 `HyperparameterTuner` and a spot-enabled `ModelTrainer` together. Parameter ranges come from `sagemaker.core.parameter`.
***

In [6]:
from sagemaker.core.parameter import ContinuousParameter, IntegerParameter
from sagemaker.core.common_utils import name_from_base
from sagemaker.train.tuner import HyperparameterTuner


# You can select from the hyperparameters supported by the model, and configure ranges of values to be searched for training the optimal model.(https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-define-ranges.html)
hyperparameter_ranges = {
    "max_depth": IntegerParameter(0, 10, scaling_type="Auto"),
    "num_round": IntegerParameter(1, 4000, scaling_type="Auto"),
    "alpha": ContinuousParameter(0, 2, scaling_type="Auto"),
    "subsample": ContinuousParameter(0.5, 1, scaling_type="Auto"),
    "min_child_weight": ContinuousParameter(0, 120, scaling_type="Auto"),
    "gamma": ContinuousParameter(0, 5, scaling_type="Auto"),
    "eta": ContinuousParameter(0.1, 0.5, scaling_type="Auto"),
}

# Increase the total number of training jobs run by AMT, for increased accuracy (and training time).
max_jobs = 3  # [exec copy] reduced from 6 to limit cost
# Change parallel training jobs run by AMT to reduce total training time, constrained by your account limits.
# if max_jobs=max_parallel_jobs then Bayesian search turns to Random.
max_parallel_jobs = 2

hp_tuner = HyperparameterTuner(
    model_trainer,
    "validation:rmse",
    hyperparameter_ranges,
    max_jobs=max_jobs,
    max_parallel_jobs=max_parallel_jobs,
    objective_type="Minimize",
    base_tuning_job_name=job_name,
)

# Launch a SageMaker Tuning job to search for the best hyperparameters
# In this case, the tuner requires a `validation` channel to emit the validation:rmse metric.
# Since we only created a `train` channel, we re-use it for validation.
validation_input = InputData(
    channel_name="validation",
    data_source="s3://{}/{}/{}".format(bucket, prefix, "train"),
    content_type="libsvm",
)
hp_tuner.tune(inputs=[train_input, validation_input])

                    INFO     Creating hyper_parameter_tuning_job resource.                       ]8;id=3555699;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3555700;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#14857\14857]8;;\

Output()

[07/13/26 16:18:23] INFO     Final Resource Status: Completed                                    ]8;id=3555706;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3555707;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#15086\15086]8;;\

## Enabling checkpointing for script mode

An additional mode of operation is to run customizable scripts as part of the training and inference jobs. See [this notebook](./xgboost_abalone_dist_script_mode.ipynb) for details on how to setup script mode. 

Here we highlight the specific changes that would enable checkpointing and use Spot instances. 

Checkpointing in script mode for SageMaker XGBoost can be performed using the convenience helpers in the `sagemaker_xgboost_container.checkpointing` module: 

- `checkpointing.train`: a convenience wrapper around `xgb.train` that loads any existing checkpoint, resumes from the last completed iteration, and registers a callback that saves a checkpoint every round. 

- `checkpointing.load_checkpoint` / `checkpointing.save_checkpoint`: the lower-level building blocks used by `checkpointing.train`. 

These read/write the checkpoint directory, which in the accompanying `abalone.py` script is set to `/opt/ml/checkpoints`. 

The relevant part of `abalone.py` looks like the following:

---------
```
from sagemaker_xgboost_container import checkpointing

CHECKPOINTS_DIR = '/opt/ml/checkpoints'   # default location for Checkpoints
train_args = dict(
    params=train_hp,
    dtrain=dtrain,
    evals=watchlist,
    num_boost_round=args.num_round,
)
bst = checkpointing.train(train_args, checkpoint_dir=CHECKPOINTS_DIR)
```

### Using ModelTrainer for XGBoost script mode

In v3 the framework-specific `sagemaker.xgboost.XGBoost` estimator is replaced by the generic `ModelTrainer`. We point it at the same XGBoost container image (retrieved via `image_uris.retrieve`, training scope) and supply our script through a `SourceCode` object (`entry_script="abalone.py"`). We also pass the IAM role, the compute configuration, and the dictionary of hyperparameters that we want to pass to our script.

In [7]:
from sagemaker.core.training.configs import SourceCode

job_name = "DEMO-xgboost-regression-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
print("Training job", job_name)
checkpoint_s3_uri = (
    "s3://{}/{}/checkpoints/{}".format(bucket, prefix, job_name) if use_spot_instances else None
)
print("Checkpoint path:", checkpoint_s3_uri)

# The script-mode training image uses the same XGBoost framework version.
script_mode_image = image_uris.retrieve(
    "xgboost", region, "1.7-1", image_scope="training"
)

xgb_script_mode_trainer = ModelTrainer(
    training_image=script_mode_image,
    role=role,
    sagemaker_session=sagemaker_session,
    source_code=SourceCode(source_dir=".", entry_script="abalone.py"),
    hyperparameters=hyperparameters,
    compute=Compute(
        instance_type=instance_type,
        instance_count=1,
        volume_size_in_gb=5,
        enable_managed_spot_training=use_spot_instances,
    ),
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=max_run,
        max_wait_time_in_seconds=max_wait,
    ),
    checkpoint_config=(
        CheckpointConfig(s3_uri=checkpoint_s3_uri) if use_spot_instances else None
    ),
    output_data_config=OutputDataConfig(
        s3_output_path="s3://{}/{}/{}/output".format(bucket, prefix, "xgboost-script-mode")
    ),
    base_job_name=job_name,
)

Training job DEMO-xgboost-regression-2026-07-13-23-18-25
Checkpoint path: s3://sagemaker-us-west-1-729646638167/sagemaker/DEMO-xgboost-spot/checkpoints/DEMO-xgboost-regression-2026-07-13-23-18-25


                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=3562942;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3562943;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#535\535]8;;\

[07/13/26 16:18:26] INFO     Role 'arn:aws:iam::729646638167:role/Admin' validated for     ]8;id=3562948;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3562949;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             training. Using it.                                                                   

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=3562954;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3562955;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/defaults.py#204\204]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=3562960;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=3562961;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             746614075791.dkr.ecr.us-west-1.amazonaws.com/sagemaker-xgboost:1.                     
                             7-1                                                                                   

Training is as simple as calling `train` on the `ModelTrainer`. This will start a SageMaker Training job that will download the data, invoke the entry point code (in the provided script file), and save any model artifacts that the script creates. In this case, the script requires a `train` and a `validation` channel. Since we only created a `train` channel, we re-use it for validation. 

In [8]:
xgb_script_mode_trainer.train(input_data_config=[train_input, validation_input])

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3562966;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3562967;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/13/26 16:18:28] INFO     Creating training_job resource.                                     ]8;id=3562972;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3562973;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31123\31123]8;;\

Output()

[07/13/26 16:20:35] INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3562978;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3562979;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             Starting training script                                                              

                    INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3562984;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3562985;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             ++ /miniconda3/bin/python3 --version                                                  

...

                    INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3564190;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3564191;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             [0]#011#011train-rmse:1.70623#011validation-rmse:1.70623                              

                    INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3564196;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3564197;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             INFO:__main__:Stored trained model at /opt/ml/model/xgboost-model                     

                    INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3564202;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3564203;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     DEMO-xgboost-regression-2026-07-13-23-18-25-20260713161826/algo-1-1 ]8;id=3564208;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3564209;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31469\31469]8;;\
                             783984744:                                                                            
                             Training Container Execution Completed                                                

[07/13/26 16:20:40] INFO     Final Resource Status: Completed                                    ]8;id=3564214;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3564215;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#31475\31475]8;;\

As previously stated, the `ModelTrainer` can also be passed to the v3 `HyperparameterTuner` object to interact with the Amazon SageMaker hyperparameter tuning APIs and create a Hyperparameter Tuning Job. Hyperparameters are automatically tuned which in most cases results in a more accurate model.

Unlike the built-in algorithm mode (whose container publishes `validation:rmse` automatically), a script-mode tuning job runs a generic training image, so we tell the tuner how to read the objective metric from the training logs via `metric_definitions`. The regex matches the `validation-rmse:<value>` lines emitted by the XGBoost checkpointing evaluation callback in `abalone.py`.

In [9]:
hp_tuner = HyperparameterTuner(
    xgb_script_mode_trainer,
    "validation:rmse",
    hyperparameter_ranges,
    metric_definitions=[
        {"Name": "validation:rmse", "Regex": "validation-rmse:([0-9\\.]+)"}
    ],
    max_jobs=max_jobs,
    max_parallel_jobs=max_parallel_jobs,
    objective_type="Minimize",
    base_tuning_job_name=job_name,
)

# Launch a SageMaker Tuning job to search for the best hyperparameters
# In this case, the tuner requires a `validation` channel to emit the validation:rmse metric.
# Since we only created a `train` channel, we re-use it for validation.
hp_tuner.tune(inputs=[train_input, validation_input])

[07/13/26 16:20:43] INFO     Creating hyper_parameter_tuning_job resource.                       ]8;id=3567460;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3567461;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#14857\14857]8;;\

Output()

[07/13/26 16:25:52] INFO     Final Resource Status: Completed                                    ]8;id=3567466;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3567467;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#15086\15086]8;;\

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)
